In [1]:
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from gensim.models import Word2Vec

import nltk
nltk.download('punkt')
from nltk.tokenize import word_tokenize

[nltk_data] Downloading package punkt to C:\Users\vishw/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [3]:
df = pd.read_csv("Car_Dataset.csv")   # make sure name matches

print(df.head())

  Make       Model  Year             Engine Fuel Type  Engine HP  \
0  BMW  1 Series M  2011  premium unleaded (required)      335.0   
1  BMW    1 Series  2011  premium unleaded (required)      300.0   
2  BMW    1 Series  2011  premium unleaded (required)      300.0   
3  BMW    1 Series  2011  premium unleaded (required)      230.0   
4  BMW    1 Series  2011  premium unleaded (required)      230.0   

   Engine Cylinders Transmission Type     Driven_Wheels  Number of Doors  \
0               6.0            MANUAL  rear wheel drive              2.0   
1               6.0            MANUAL  rear wheel drive              2.0   
2               6.0            MANUAL  rear wheel drive              2.0   
3               6.0            MANUAL  rear wheel drive              2.0   
4               6.0            MANUAL  rear wheel drive              2.0   

                         Market Category Vehicle Size Vehicle Style  \
0  Factory Tuner,Luxury,High-Performance      Compact         C

In [5]:
# =====================================
# 3. DATA PREPROCESSING (IMPORTANT)
# Convert structured data → text
# =====================================
df = df.fillna("")   # handle missing values

df['text'] = df.astype(str).agg(' '.join, axis=1)

print("\nSample Text Data:")
print(df['text'].head())



Sample Text Data:
0    BMW 1 Series M 2011 premium unleaded (required...
1    BMW 1 Series 2011 premium unleaded (required) ...
2    BMW 1 Series 2011 premium unleaded (required) ...
3    BMW 1 Series 2011 premium unleaded (required) ...
4    BMW 1 Series 2011 premium unleaded (required) ...
Name: text, dtype: object


In [6]:
# =====================================
# 4. BAG OF WORDS (COUNT OCCURRENCE)
# =====================================
cv = CountVectorizer()

X_bow = cv.fit_transform(df['text'])

print("\nBoW Shape:", X_bow.shape)
print("Sample Features:", cv.get_feature_names_out()[:10])


BoW Shape: (11914, 7315)
Sample Features: ['10' '100' '1001' '100100' '100600' '100660' '100800' '101' '1013'
 '101300']


In [7]:
# =====================================
# 5. NORMALIZED BAG OF WORDS
# =====================================
X_bow_array = X_bow.toarray()

# Normalize (avoid divide by zero)
row_sums = X_bow_array.sum(axis=1, keepdims=True)
row_sums[row_sums == 0] = 1

X_bow_norm = X_bow_array / row_sums

print("\nNormalized BoW (first row):")
print(X_bow_norm[0])


Normalized BoW (first row):
[0. 0. 0. ... 0. 0. 0.]


In [8]:
# =====================================
# 6. TF-IDF
# =====================================
tfidf = TfidfVectorizer()

X_tfidf = tfidf.fit_transform(df['text'])

print("\nTF-IDF Shape:", X_tfidf.shape)
print("Sample Features:", tfidf.get_feature_names_out()[:10])


TF-IDF Shape: (11914, 7315)
Sample Features: ['10' '100' '1001' '100100' '100600' '100660' '100800' '101' '1013'
 '101300']


In [9]:
# =====================================
# 7. WORD2VEC
# =====================================

# Tokenization
tokenized_data = [word_tokenize(text.lower()) for text in df['text']]

# Train model
w2v_model = Word2Vec(
    sentences=tokenized_data,
    vector_size=100,
    window=5,
    min_count=1,
    workers=4
)

In [10]:
# Example: word vector
print("\nWord2Vec Vector for 'bmw':")
print(w2v_model.wv['bmw'])


Word2Vec Vector for 'bmw':
[-1.4851481  -0.03230631 -0.10648447 -0.06695622 -0.43480858 -0.06563304
 -0.5700949   0.19547167 -0.51672834 -0.4360356   1.5517457  -0.6732817
  1.1238337  -0.02245994  0.64942837  0.2353654   1.6615864  -0.68347347
  0.57438856 -1.260603    0.38597846  0.11073476  0.00290001  0.39867938
  0.7283514   0.58493817  0.6704175  -0.6056035   1.4518596   0.32586345
  0.47004518  0.91134644 -0.29786718  1.106328    1.2465692   0.07791362
  0.36543286 -0.0753612   0.11156064  0.5255062  -0.28016135  0.32145625
  1.1788851   0.5151994   1.1695014  -1.2734066   0.46376356  0.4365513
  0.06030238 -0.94156736 -0.5023837   0.8405018  -1.1684144   1.9716474
  1.1577473  -0.7747245   0.8375775   0.7053691   0.94536996  1.6314692
  0.39287022 -0.9152742   0.8299211   1.5914605  -0.32056063 -1.3585153
 -0.13279885 -1.3194183   0.3944773   1.4813751   1.0627754   0.2812481
  0.469719   -0.02678129 -0.07812952  0.67760795 -0.42408645 -0.06016631
  0.3439605  -1.1842552  -0.5

In [11]:
# Similar words
print("\nWords similar to 'bmw':")
print(w2v_model.wv.most_similar('bmw'))


Words similar to 'bmw':
[('porsche', 0.9851566553115845), ('200t', 0.9798590540885925), ('s-class', 0.9789897799491882), ('continental', 0.9775979518890381), ('e-class', 0.976863443851471), ('panamera', 0.9753484725952148), ('gt', 0.9744418263435364), ('sl-class', 0.9728712439537048), ('450h', 0.9720711708068848), ('rover', 0.9715623259544373)]


In [12]:
# =====================================
# 8. DONE
# =====================================
print("\nAll tasks completed successfully ✅")


All tasks completed successfully ✅


In [84]:
embeddings

array([[-0.01106373,  0.00585934,  0.01540476, ..., -0.03831561,
         0.01001132,  0.00379507],
       [-0.01955247,  0.00990319,  0.01476101, ..., -0.03403803,
         0.00382115,  0.00643379],
       [-0.0184865 ,  0.00986475,  0.01654671, ..., -0.04207258,
         0.0108053 ,  0.00743209],
       ...,
       [-0.01277368,  0.00704898,  0.00513679, ..., -0.02371885,
         0.00932181, -0.00200806],
       [-0.01277368,  0.00704898,  0.00513679, ..., -0.02371885,
         0.00932181, -0.00200806],
       [-0.0184865 ,  0.00986475,  0.01654671, ..., -0.04207258,
         0.0108053 ,  0.00743209]])